In [ ]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.metrics import enrichment_factor
from surrogate_model.optuna import RECALL_TOP_1, make_objective

In [ ]:
FEATURES = "data/candidates.10k.parquet"
LABELS = "data/1L83.1L83:p2rank:2.10k.parquet"

RANDOM_SEED = 1000

In [ ]:
features = pl.read_parquet(FEATURES)

features = features.filter(
    (pl.col("parse_ok"))
    & (pl.col("error") == "SUCCESS")
    & (pl.col("conversion_error") == "SUCCESS")
)

cns_mpo_schema = pl.Struct(
    [
        pl.Field("clogp", pl.Float64),
        pl.Field("clogd", pl.Float64),
        pl.Field("tpsa", pl.Float64),
    ]
)

features = features.with_columns(
    pl.col("cns_mpo_components").str.json_decode(cns_mpo_schema)
).unnest("cns_mpo_components")

In [ ]:
features = features.drop(
    ["smiles", "parse_ok", "pains_flags", "error", "conversion_error"]
)

In [ ]:
labels = pl.read_parquet(LABELS)
labels = labels["catalog_id", "affinity_kcal_mol"]

In [ ]:
df = features.join(labels, on="catalog_id", how="inner")

In [ ]:
# df = df.sample(1000)

In [ ]:
FEATURE_NAMES = ["heavy_atom_count", "molecular_weight", "clogp", "clogd", "tpsa"]

LABEL_NAME = "affinity_kcal_mol"

In [ ]:
x = df.select(FEATURE_NAMES).to_numpy()
x = np.hstack([x, np.array(df["morgan_fp"].to_list())])

In [ ]:
y = df[LABEL_NAME].to_numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_SEED
)

In [ ]:
study = optuna.create_study(direction="minimize")

In [ ]:
study.optimize(
    make_objective(
        X_train, y_train, 5, primary_metric=RECALL_TOP_1, random_seed=RANDOM_SEED
    ),
    n_trials=30,
    show_progress_bar=True,
)

In [ ]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(
    X_train,
    y_train,
    eval_X=X_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

In [ ]:
y_pred = final_model.predict(X_test)

In [ ]:
enrichment = enrichment_factor(y_pred, y_test, 0.05)

In [ ]:
enrichment

# Results

| --- | --- | --- |
| dataset size | trials | enrichment |
| 1000 | 30 | 4.0 |
